In [1]:
# Chit 30: CBOW using a sample text corpus
# Step 1: Prepare a text corpus
import re
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense

with open("document3.txt", encoding="utf-8") as file:
    text = file.read()

text = re.sub(r"[^a-z\s]", " ", text.lower())
words = text.split()
vocabulary = sorted(set(words))
word_to_id = {word: i for i, word in enumerate(vocabulary)}
id_to_word = {i: word for word, i in word_to_id.items()}
print("Vocabulary:", vocabulary)

Vocabulary: ['a', 'an', 'and', 'appear', 'appearance', 'are', 'at', 'be', 'before', 'between', 'both', 'can', 'cases', 'comparisons', 'context', 'contrast', 'covid', 'days', 'difference', 'difficult', 'direct', 'does', 'driver', 'estimated', 'estimates', 'faster', 'first', 'for', 'from', 'further', 'generated', 'has', 'higher', 'hours', 'however', 'illness', 'important', 'in', 'incubation', 'individual', 'infected', 'infection', 'infections', 'influenza', 'interval', 'is', 'learning', 'major', 'making', 'means', 'median', 'more', 'not', 'number', 'of', 'one', 'onset', 'or', 'people', 'period', 'point', 'potentially', 'pre', 'present', 'prior', 'reproductive', 'secondary', 'serial', 'shed', 'shorter', 'specific', 'speed', 'spread', 'successive', 'symptom', 'symptomatic', 'symptoms', 'than', 'that', 'the', 'there', 'this', 'time', 'to', 'transmission', 'two', 'understood', 'very', 'virus', 'viruses', 'we', 'while', 'who']


In [2]:
# Step 2: Make context and target pairs
window_size = 2
contexts, targets = [], []
for i in range(window_size, len(words) - window_size):
    context_words = words[i-window_size:i] + words[i+1:i+window_size+1]
    contexts.append([word_to_id[word] for word in context_words])
    targets.append(word_to_id[words[i]])

X = np.array(contexts, dtype="int32")
y = np.array(targets, dtype="int32")
for i in range(min(5, len(X))):
    print([id_to_word[index] for index in X[i]], "->", id_to_word[y[i]])

['the', 'speed', 'transmission', 'is'] -> of
['speed', 'of', 'is', 'an'] -> transmission
['of', 'transmission', 'an', 'important'] -> is
['transmission', 'is', 'important', 'point'] -> an
['is', 'an', 'point', 'of'] -> important


In [3]:
# Step 3: Train the CBOW network
embedding_size = 16
model = Sequential([
    Input(shape=(2 * window_size,)),
    Embedding(input_dim=len(vocabulary), output_dim=embedding_size),
    GlobalAveragePooling1D(),
    Dense(len(vocabulary), activation="softmax")
])
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
history = model.fit(X, y, epochs=100, verbose=0)
print("Final training loss:", round(history.history["loss"][-1], 4))

Final training loss: 2.9378


In [4]:
# Step 4: Predict a target word
sample_context = X[0]
prediction = model.predict(sample_context.reshape(1, -1), verbose=0)
predicted_word = id_to_word[int(np.argmax(prediction[0]))]
print("Context:", [id_to_word[index] for index in sample_context])
print("Predicted target:", predicted_word)

Context: ['the', 'speed', 'transmission', 'is']
Predicted target: of
